# Howen VSS Web API — Full Explorer
**Run each cell in order. Main exports are under `output/` (JSON; Section 3b can write Excel).**

Per **Howen VSS Web API** (section 3.9 `apiFindAllByTime.action`, section 3.20.7 `findDriverInfoByFleet.action`): fleet display names come from the driver/fleet API; historical alarms are queried per `deviceID` and time range.

| Section | What it does |
|---|---|
| 0 | Setup & Config |
| 1 | Login → Get Token |
| 2 | All Devices / Units |
| 3 | One JSON per **VSS group** (`fleetname` / `fleetid` — same as the fleet sidebar) |
| 3b | Onboarding Excel: filter by **fleetname** (e.g. PONTY PRIDD; `createtime`, full columns) |
| 4 | Live GPS Status of all Devices |
| 5b | `DHL_all_vehicles.json`, `DHL_alerts_2026-04.json`, **`DHL_Alerts.xlsx`** (April window) |
| 6 | Output file summary |

---
## ⚙️ Section 0 — Setup & Config

In [1]:
# Install dependencies if needed
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'requests', 'pandas', 'openpyxl', '-q'])

CompletedProcess(args=['C:\\Users\\YOGA\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', '-m', 'pip', 'install', 'requests', 'pandas', 'openpyxl', '-q'], returncode=0)

In [2]:
import requests
import hashlib
import json
import re
import time
import pandas as pd
import os
from collections import defaultdict
from IPython.display import display

requests.packages.urllib3.disable_warnings()

# ─── EDIT THESE ───────────────────────────────────
BASE_URL   = "https://vss.controltech-ea.com"
USERNAME   = "musyoka@controltech-ea.com"
PASSWORD   = "Kenya123+"
BATCH_SIZE = 20          # devices per GPS request
OUTPUT_DIR = "output"

# Optional: map fleet GUID → display name if the API name is wrong or missing
FLEET_ID_TO_NAME = {
    # "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx": "DHL",
}

# Window for DHL alerts export (section 3.9 API — keep ranges reasonable for server load)
ALL_ALERTS_BEGIN = "2026-01-01 00:00:00"
ALL_ALERTS_END   = "2026-12-31 23:59:59"

# DHL-only report (April 2026, inclusive)
DHL_REPORT_LABEL = "DHL"
DHL_ALERTS_BEGIN = "2026-04-01 00:00:00"
DHL_ALERTS_END   = "2026-04-30 23:59:59"

# Section 3b: Excel onboarding — rows where API fleetname equals EXCEL_REPORT_GROUP_NAME
EXCEL_REPORT_GROUP_NAME = "PONTY PRIDD"

ALARM_PAGE_SIZE = 200
ALARM_REQ_PAUSE_S = 0.05   # small pause between per-device alarm calls
# ──────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Shared state across cells
TOKEN   = None
PID     = None
DEVICES = []
FLEET_LABEL_BY_ID = {}   # fleet_id → display name (filled in Section 3)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 300)
pd.set_option('display.max_colwidth', 35)
pd.set_option('display.max_rows', 200)

def save_json(data, filename, quiet=False):
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    if not quiet:
        print(f'  Saved → {path}')
    return path

def customer_key_from_device_name(name):
    """Customer key from VSS devicename, e.g. 'EXPEDITERS - KBK 863V' -> 'EXPEDITERS'."""
    s = (name or '').replace('\u00a0', ' ').strip()
    if not s:
        return 'UNKNOWN'
    if ' - ' in s:
        key = s.split(' - ', 1)[0].strip()
    else:
        parts = re.split(r'[\s\-_/|]+', s, maxsplit=1)
        key = (parts[0] if parts else s).strip()
    key = re.sub(r'^[^\w&]+|[^\w&]+$', '', key)
    return (key or 'UNKNOWN').upper()

def device_in_dhl_scope(device_dict):
    """True if device counts as DHL for roll-up + alarm export (fleet, resolved label, or devicename prefix)."""
    label = (DHL_REPORT_LABEL or 'DHL').strip()
    if not label:
        return False
    want = label.lower()
    fn = (device_dict.get('fleetname') or '').strip().lower()
    if fn == want or fn.startswith(want + '-') or fn.startswith(want + ' '):
        return True
    fid = device_dict.get('fleetid')
    if fid:
        fl = (FLEET_LABEL_BY_ID.get(fid) or '').strip().lower()
        if fl == want or fl.startswith(want + '-') or fl.startswith(want + ' '):
            return True
    return customer_key_from_device_name(device_dict.get('devicename') or '') == label.upper()

def status_label(code):
    m = {-1:'Offline',0:'Unknown',1:'Ethernet',2:'WiFi',
         3:'2G',4:'3G',5:'4G',6:'5G',7:'WiFi+Mobile',8:'Cable+Mobile'}
    try: return m.get(int(code), f'Code({code})')
    except Exception:
        return str(code)

def safe_customer_filename(name):
    name = (name or '').strip() or 'customer'
    name = re.sub(r'[<>:"/\\|?*\n\r\t]', '_', name)
    name = name.strip('. ') or 'customer'
    return name[:120]

def resolve_fleet_name(fleet_id, sample_device_id):
    """Uses section 3.20.7 findDriverInfoByFleet.action — returns fleetname for one device in that fleet."""
    if fleet_id in FLEET_ID_TO_NAME:
        return FLEET_ID_TO_NAME[fleet_id]
    if not TOKEN or not sample_device_id:
        return f'Fleet_{str(fleet_id)[:8]}'
    url = f'{BASE_URL}/vss/driver/findDriverInfoByFleet.action'
    try:
        r = requests.post(url, json={'token': TOKEN, 'deviceId': str(sample_device_id), 'scheme': 'https'},
                          verify=False, timeout=45)
        j = r.json()
        if j.get('status') == 10000 and j.get('data'):
            for block in j['data']:
                fn = (block.get('fleetname') or '').strip()
                if fn:
                    return fn
    except Exception:
        pass
    return f'Fleet_{str(fleet_id)[:8]}'

def fetch_alarms_for_device(device_id, begin_time, end_time, alarm_type=''):
    """Section 3.9 QueryHistoryAlarm — POST .../vss/alarm/apiFindAllByTime.action"""
    url = f'{BASE_URL}/vss/alarm/apiFindAllByTime.action'
    out, page = [], 1
    while True:
        payload = {
            'token': TOKEN,
            'deviceID': str(device_id),
            'pageNum': str(page),
            'pageCount': str(ALARM_PAGE_SIZE),
            'beginTime': begin_time,
            'endTime': end_time,
            'alarmType': alarm_type,
        }
        r = requests.post(url, json=payload, verify=False, timeout=120)
        j = r.json()
        if j.get('status') != 10000:
            return out, j.get('msg') or j.get('status')
        data = j.get('data') or {}
        rows = data.get('dataList') or []
        out.extend(rows)
        total_num = int(data.get('totalNum') or 1)
        if page >= total_num or not rows:
            break
        page += 1
    return out, None

print('Setup complete. Proceed to Section 1.')

Setup complete. Proceed to Section 1.


---
## 🔑 Section 1 — Login & Get Token

In [3]:
# ── LOGIN ──────────────────────────────────────────
url    = f"{BASE_URL}/vss/user/apiLogin.action"
md5_pw = hashlib.md5(PASSWORD.encode()).hexdigest()

print(f'URL      : {url}')
print(f'Username : {USERNAME}')
print(f'MD5 Hash : {md5_pw}')
print()

resp = requests.post(url, json={'username': USERNAME, 'password': md5_pw},
                     verify=False, timeout=30)
data = resp.json()

if data.get('status') == 10000:
    TOKEN = data['data']['token']
    PID   = data['data']['pid']

    result = {'status':'success','token':TOKEN,'pid':PID,
              'username':USERNAME,'base_url':BASE_URL}
    save_json(result, '01_token.json')

    print(f'\n✅ Login SUCCESS')
    print(f'Token : {TOKEN}')
    print(f'PID   : {PID[:60]}...')
else:
    print(f'❌ Login FAILED — {data.get("msg")}  (code {data.get("status")})')

URL      : https://vss.controltech-ea.com/vss/user/apiLogin.action
Username : musyoka@controltech-ea.com
MD5 Hash : 3a5baa6364bdba0d10f7da07a158a711

❌ Login FAILED — Login too frequently  (code 10082)


---
## 🚛 Section 2 — All Devices / Units

In [ ]:
# ── FETCH ALL DEVICES ──────────────────────────────
url  = f"{BASE_URL}/vss/vehicle/findAll.action"
resp = requests.post(url,
                     json={'token':TOKEN,'pageNum':-1,'pageCount':-1,'keyword':''},
                     verify=False, timeout=30)
data = resp.json()

if data.get('status') != 10000:
    print(f'❌ Failed: {data.get("msg")}')
else:
    DEVICES = data['data']['dataList']
    total   = data['data']['totalCount']

    save_json(DEVICES, '02_devices.json')
    print(f'\n✅ Total devices returned : {total}')

In [ ]:
# ── BUILD DATAFRAME ─────────────────────────────────
rows = []
for d in DEVICES:
    rows.append({
        'device_id'    : d.get('deviceno',''),
        'device_name'  : d.get('devicename',''),
        'device_type'  : d.get('devicetype',''),
        'fleet_id'     : d.get('fleetid',''),
        'online_status': status_label(d.get('accessmode',-1)),
        'last_online'  : d.get('lastonlinetime',''),
        'last_offline' : d.get('lastofflinetime',''),
        'longitude'    : d.get('longitude',''),
        'latitude'     : d.get('latitude',''),
        'sim'          : d.get('sim',''),
        'imei'         : d.get('imei',''),
        'plate_no'     : d.get('plateno',''),
        'channels'     : d.get('videoencodernumber',''),
        'firmware'     : d.get('appVersion',''),
        'created'      : d.get('createtime',''),
        'today_mb'     : round(d.get('dayUsed',0)/1024/1024, 2),
        'month_mb'     : round(d.get('monthUsed',0)/1024/1024, 2),
    })

df_devices = pd.DataFrame(rows)
if len(rows):
    save_json(rows, '02_devices_flat.json')

print(f'\nDevices DataFrame ({len(df_devices)} rows):')
display(df_devices)

In [ ]:
# ── ONLINE STATUS SUMMARY ───────────────────────────
print('Online status breakdown:')
if df_devices.empty or 'online_status' not in df_devices.columns:
    print('  (no rows — run login + device fetch first)')
else:
    display(df_devices['online_status'].value_counts().rename_axis('Status').reset_index(name='Count'))

---
## Section 3 — One JSON per VSS group (fleetname / fleetid)

In [ ]:
# ── GROUP DEVICES BY FLEET (VSS sidebar: fleetname / fleetid) ─
fleet_map = defaultdict(list)
for d in DEVICES:
    fid = d.get('fleetid') or 'NO_FLEET'
    fleet_map[fid].append({
        'device_id'    : d.get('deviceno',''),
        'device_name'  : d.get('devicename',''),
        'fleet_id'     : d.get('fleetid', ''),
        'fleet_name'   : (d.get('fleetname') or '').strip(),
        'device_type'  : d.get('devicetype',''),
        'online_status': status_label(d.get('accessmode',-1)),
        'last_online'  : d.get('lastonlinetime',''),
        'latitude'     : d.get('latitude',''),
        'longitude'    : d.get('longitude',''),
        'plate_no'     : d.get('plateno',''),
        'sim'          : d.get('sim',''),
    })

fleet_map = dict(fleet_map)

# Label each fleet: device fleetname first, then FLEET_ID_TO_NAME, then driver API
fleetname_first = {}
for d in DEVICES:
    fid = d.get('fleetid') or 'NO_FLEET'
    fn = (d.get('fleetname') or '').strip()
    if fn and fid not in fleetname_first:
        fleetname_first[fid] = fn

FLEET_LABEL_BY_ID = {}
for fid, units in fleet_map.items():
    if fid in FLEET_ID_TO_NAME:
        FLEET_LABEL_BY_ID[fid] = FLEET_ID_TO_NAME[fid]
    elif fid in fleetname_first:
        FLEET_LABEL_BY_ID[fid] = fleetname_first[fid]
    else:
        sample = units[0]['device_id'] if units else None
        FLEET_LABEL_BY_ID[fid] = resolve_fleet_name(fid, sample)

save_json({'fleets': fleet_map, 'fleet_display_name': FLEET_LABEL_BY_ID}, '03_fleets_by_id.json', quiet=True)
print(f'Total fleets/groups: {len(fleet_map)} (03_fleets_by_id.json)')

In [ ]:
# ── GROUPS SUMMARY (VSS fleet — matches sidebar) ────
summary_list = [
    {
        'group_name': FLEET_LABEL_BY_ID.get(fid, fid),
        'fleet_id': fid,
        'unit_count': len(units),
        'device_ids': [u['device_id'] for u in units if u.get('device_id')],
    }
    for fid, units in sorted(
        fleet_map.items(),
        key=lambda x: (FLEET_LABEL_BY_ID.get(x[0], str(x[0])).upper(), x[0]),
    )
]
save_json(summary_list, '03_groups_summary.json', quiet=True)
save_json(summary_list, '03_customers_summary.json', quiet=True)
print('Saved 03_groups_summary.json (%s groups)' % len(summary_list))


In [ ]:
# Preview removed (use 03_groups_summary.json or per-group JSON files in output/)

In [ ]:
# ── ONE JSON PER VSS GROUP (filename = fleet / sidebar name) ─
used_names = set()
n_saved = 0
for fid, units in fleet_map.items():
    label = FLEET_LABEL_BY_ID.get(fid, fid)
    base = safe_customer_filename(label)
    fname = f'{base}.json'
    n = 2
    while fname.lower() in used_names:
        fname = f'{base}_{n}.json'
        n += 1
    used_names.add(fname.lower())
    ucount = len(units)
    payload = {
        'title': f'{label} ({ucount} units)',
        'group_name': label,
        'fleet_id': fid,
        'unit_count': ucount,
        'units': units,
    }
    save_json(payload, fname, quiet=True)
    n_saved += 1

print('Saved %s group JSON file(s) to %s/' % (n_saved, OUTPUT_DIR))

In [ ]:
# ── DHL: all vehicles in one JSON (fleet + devicename — run Section 3 first for best fleet labels)
if not DEVICES:
    print('Run Section 2 first — load DEVICES.')
else:
    if not FLEET_LABEL_BY_ID:
        print('Tip: run Section 3 before this cell so fleet labels resolve; devicename DHL- prefix still works.')
    dhl_all = []
    for d in DEVICES:
        if device_in_dhl_scope(d):
            dhl_all.append({
                'device_id': d.get('deviceno', ''),
                'device_name': d.get('devicename', ''),
                'fleet_id': d.get('fleetid', ''),
                'fleet_name': (d.get('fleetname') or '').strip(),
                'device_type': d.get('devicetype', ''),
                'online_status': status_label(d.get('accessmode', -1)),
                'last_online': d.get('lastonlinetime', ''),
                'latitude': d.get('latitude', ''),
                'longitude': d.get('longitude', ''),
                'plate_no': d.get('plateno', ''),
                'sim': d.get('sim', ''),
            })
    save_json({
        'title': '%s (%s units)' % (DHL_REPORT_LABEL, len(dhl_all)),
        'group_name': DHL_REPORT_LABEL,
        'report_label': DHL_REPORT_LABEL,
        'unit_count': len(dhl_all),
        'units': dhl_all,
    }, 'DHL_all_vehicles.json')
    print('Saved DHL_all_vehicles.json (%s units)' % len(dhl_all))


In [ ]:
# ── DHL Device Maintenance report (Excel) ──────────────────────────────
# This uses fields available from /vss/vehicle/findAll (DEVICES) and matches the "Device Maintenance" idea:
# online/offline, last online, SIM/plate presence, etc.
if not DEVICES:
    print('Run Section 2 first — load DEVICES.')
else:
    dhl_devices = [d for d in DEVICES if device_in_dhl_scope(d)]
    if not dhl_devices:
        print('No devices in DHL scope (check DHL_REPORT_LABEL and run Section 3).')
    else:
        try:
            dhl_dev_df = pd.json_normalize(dhl_devices, sep='_')
        except Exception:
            dhl_dev_df = pd.DataFrame(dhl_devices)

        # Core columns first (only keep ones that exist)
        core = [
            'deviceno',
            'devicename',
            'fleetname',
            'fleetid',
            'accessmode',
            'lastonlinetime',
            'createtime',
            'modifytime',
            'plateno',
            'sim',
            'imei',
            'iccid',
            'devicetype',
            'deviceModel',
            'appVersion',
            'installDate',
            'remarks',
        ]
        cols = [c for c in core if c in dhl_dev_df.columns] + [c for c in dhl_dev_df.columns if c not in core]
        dhl_dev_df = dhl_dev_df[cols]

        # Derived: online status label
        if 'accessmode' in dhl_dev_df.columns:
            dhl_dev_df['online_status'] = dhl_dev_df['accessmode'].apply(status_label)

        # Parse last online time where possible
        last_online_ts = None
        if 'lastonlinetime' in dhl_dev_df.columns:
            last_online_ts = pd.to_datetime(dhl_dev_df['lastonlinetime'], errors='coerce')
            dhl_dev_df['_last_online_ts'] = last_online_ts

        # Filters
        missing_sim_df = dhl_dev_df[dhl_dev_df.get('sim', '').astype(str).str.strip().eq('')].copy() if 'sim' in dhl_dev_df.columns else pd.DataFrame()
        missing_plate_df = dhl_dev_df[dhl_dev_df.get('plateno', '').astype(str).str.strip().eq('')].copy() if 'plateno' in dhl_dev_df.columns else pd.DataFrame()

        offline_long_df = pd.DataFrame()
        if last_online_ts is not None:
            cutoff = pd.Timestamp.now(tz=None) - pd.Timedelta(days=7)
            offline_long_df = dhl_dev_df[last_online_ts.isna() | (last_online_ts < cutoff)].copy()

        status_summary = pd.DataFrame()
        if 'online_status' in dhl_dev_df.columns:
            status_summary = (
                dhl_dev_df['online_status']
                .value_counts()
                .rename_axis('online_status')
                .reset_index(name='count')
            )

        out_xlsx = os.path.join(OUTPUT_DIR, 'DHL_Device_Maintenance.xlsx')
        with pd.ExcelWriter(out_xlsx, engine='openpyxl') as w:
            dhl_dev_df.to_excel(w, sheet_name='Devices', index=False)
            if not status_summary.empty:
                status_summary.to_excel(w, sheet_name='OnlineSummary', index=False)
            if not offline_long_df.empty:
                offline_long_df.to_excel(w, sheet_name='Offline_7d_plus', index=False)
            if not missing_sim_df.empty:
                missing_sim_df.to_excel(w, sheet_name='MissingSIM', index=False)
            if not missing_plate_df.empty:
                missing_plate_df.to_excel(w, sheet_name='MissingPlate', index=False)

        print('DHL devices:', len(dhl_dev_df), '| Excel ->', out_xlsx)


---
## Section 3b — VSS onboarding report (Excel)

Includes units whose **`fleetname`** matches **`EXCEL_REPORT_GROUP_NAME`** (same string as the fleet sidebar). Run **Section 3** first so missing names can fall back to **`FLEET_LABEL_BY_ID`**. The API field **`createtime`** is when the device was **created on the VSS platform** (onboarding). **`installDate`** is install date when the server provides it.

**Output:** `output/<EXCEL_REPORT_GROUP_NAME>_VSS_onboarding_report.xlsx`


In [ ]:
# Section 3b — Excel: fleet created / onboarded on VSS (createtime + all columns)
if not DEVICES:
    print("Run Section 2 first — load DEVICES.")
else:
    want = (EXCEL_REPORT_GROUP_NAME or "").strip().lower()
    rows = []
    for d in DEVICES:
        fn = (d.get("fleetname") or "").strip().lower()
        if fn == want:
            rows.append(d)
        elif not (d.get("fleetname") or "").strip():
            fid = d.get("fleetid")
            if fid and (FLEET_LABEL_BY_ID.get(fid) or "").strip().lower() == want:
                rows.append(d)

    title = (EXCEL_REPORT_GROUP_NAME or "report").strip()
    ponty_pridd_onboarding_df = pd.DataFrame()

    if not rows:
        print("No devices matched EXCEL_REPORT_GROUP_NAME:", repr(EXCEL_REPORT_GROUP_NAME))
    else:
        try:
            df_full = pd.json_normalize(rows, sep="_")
        except Exception:
            df_full = pd.DataFrame(rows)

        onboard_first = [
            "deviceno",
            "devicename",
            "fleetname",
            "fleetid",
            "createtime",
            "installDate",
            "modifytime",
            "devicetype",
            "deviceModel",
            "appVersion",
            "plateno",
            "vehicletype",
            "sim",
            "imei",
            "iccid",
            "lastonlinetime",
            "remarks",
        ]
        preferred = [c for c in onboard_first if c in df_full.columns]
        rest = [c for c in df_full.columns if c not in preferred]
        ponty_pridd_onboarding_df = df_full[preferred + rest]
        if "createtime" in ponty_pridd_onboarding_df.columns:
            ponty_pridd_onboarding_df = ponty_pridd_onboarding_df.sort_values(
                "createtime", na_position="last"
            )

    base = safe_customer_filename(title)
    xlsx = os.path.join(OUTPUT_DIR, "%s_VSS_onboarding_report.xlsx" % base)
    ponty_pridd_onboarding_df.to_excel(xlsx, sheet_name="Onboarding", index=False, engine="openpyxl")
    print(
        len(ponty_pridd_onboarding_df),
        "units,",
        ponty_pridd_onboarding_df.shape[1],
        "columns ->",
        xlsx,
    )
    display(ponty_pridd_onboarding_df.head())


---
## Section 5 — DHL historical alarms & DHL April 2026

Uses **section 3.9** `POST /vss/alarm/apiFindAllByTime.action` per device (with paging). For heavy date ranges the manual recommends narrow windows and monitoring server load.

Outputs:

- `output/DHL_all_vehicles.json` — every device in **DHL scope** (exact / `DHL-…` fleetnames, resolved fleet label after Section 3, or **`DHL` devicename prefix**).
- `output/DHL_alerts_2026-04.json` — merged alarm rows for that scope over **`DHL_ALERTS_BEGIN` … `DHL_ALERTS_END`** (April 2026 by default).
- `output/DHL_Alerts.xlsx` — same alarms as the JSON (**Alarms** sheet; **Errors** sheet if any per-device API errors).

Adjust `DHL_REPORT_LABEL` / `DHL_ALERTS_*` in the setup cell for other months. Run **Section 3** first for correct fleet labels.

In [ ]:
# Section 5b — DHL alarms for window DHL_ALERTS_BEGIN … DHL_ALERTS_END (April by default) + Excel
if not TOKEN:
    print('No token — run Section 1 login first.')
elif not DEVICES:
    print('No devices — run Sections 2 and 3 first.')
else:
    dhl_ids = []
    for d in DEVICES:
        if device_in_dhl_scope(d):
            x = d.get('deviceno')
            if x:
                dhl_ids.append(str(x))
    dhl_ids = list(dict.fromkeys(dhl_ids))

    xlsx_path = os.path.join(OUTPUT_DIR, 'DHL_Alerts.xlsx')

    if not dhl_ids:
        print('No devices in DHL scope (fleet / devicename — run Section 3 for fleet labels).')
        save_json({
            'customer': DHL_REPORT_LABEL,
            'beginTime': DHL_ALERTS_BEGIN,
            'endTime': DHL_ALERTS_END,
            'devices_queried': [],
            'alarm_count': 0,
            'note': 'No devices matched DHL scope.',
            'alarms': [],
        }, 'DHL_alerts_2026-04.json')
        with pd.ExcelWriter(xlsx_path, engine='openpyxl') as w:
            pd.DataFrame({'note': ['No DHL-scoped devices; no alarm rows.']}).to_excel(w, sheet_name='Alarms', index=False)
        print('Excel ->', xlsx_path)
    else:
        dhl_rows, dhl_errs = [], []
        print('DHL alerts: %s to %s | devices: %s' % (DHL_ALERTS_BEGIN, DHL_ALERTS_END, len(dhl_ids)))
        for dev in dhl_ids:
            rows, err = fetch_alarms_for_device(dev, DHL_ALERTS_BEGIN, DHL_ALERTS_END)
            if err:
                dhl_errs.append({'deviceID': dev, 'msg': str(err)})
            for r in rows:
                rr = dict(r) if isinstance(r, dict) else {'raw': r}
                rr['_queried_device_id'] = dev
                dhl_rows.append(rr)
            time.sleep(ALARM_REQ_PAUSE_S)

        save_json({
            'customer': DHL_REPORT_LABEL,
            'beginTime': DHL_ALERTS_BEGIN,
            'endTime': DHL_ALERTS_END,
            'devices_queried': dhl_ids,
            'alarm_count': len(dhl_rows),
            'errors': dhl_errs,
            'alarms': dhl_rows,
        }, 'DHL_alerts_2026-04.json')

        try:
            dhl_alarm_df = pd.json_normalize(dhl_rows, sep='_') if dhl_rows else pd.DataFrame()
        except Exception:
            dhl_alarm_df = pd.DataFrame(dhl_rows)
        with pd.ExcelWriter(xlsx_path, engine='openpyxl') as w:
            dhl_alarm_df.to_excel(w, sheet_name='Alarms', index=False)
            if dhl_errs:
                pd.DataFrame(dhl_errs).to_excel(w, sheet_name='Errors', index=False)
        print('DHL alarm rows:', len(dhl_rows), '| Excel ->', xlsx_path)


---
## 📁 Output File Summary

In [ ]:
# ── LIST ALL OUTPUT FILES ───────────────────────────
import os
files = []
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(path)
    files.append({'file': f, 'size_kb': round(size/1024, 1)})

df_files = pd.DataFrame(files)
print(f'📁 Files in output/ folder:')
display(df_files)